# ML-10: Content Action Playbook

**Name:** Abdullah Hasan Shah

**Track:** Machine Learning

**Lane:** Content Refresh

## Objective

The objective of this notebook is to transform the validated machine learning model into a practical content action playbook. The model predictions are converted into prioritized actions that content teams can review before updating web pages.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

pd.set_option("display.max_columns", None)

In [2]:
df = pd.read_csv("C:/Users/ok/Documents/FlyRank Internship/Week 1/flyrank-ml-internship-starter-main/data/processed/refresh_feature_vector.csv")

df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,provider_used,model_used,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,engaged_sessions_90d,ai_sessions_90d,scroll_events_90d,days_with_impressions,days_with_sessions,impressions_last_30d,clicks_last_30d,sessions_last_30d,impressions_prev_30d,clicks_prev_30d,sessions_prev_30d,content_age_days,age_tier,age_tier_order,days_since_last_update,freshness_tier,word_count_tier,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,is_declining_label,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,has_clicks,has_ai_sessions,measurable_opportunity
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,unknown,gemini-2.5-flash,3803,29,22,17,16,1,0,1,88,13,578,2,2,987,13,9,187,181-365,5,20,0-30,2000-3500,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4,1,8.243808,3.401197,2.890372,0.0,1,0,1
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,unknown,gemini-3-flash-preview,15320,7,10,9,9,0,0,1,88,9,2501,2,3,5915,1,2,445,365+,6,25,0-30,2000-3500,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7,1,9.636980,2.079442,2.302585,0.0,1,0,1
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,unknown,gemini-2.5-flash,12581,11,14,11,11,0,0,4,88,11,2382,1,1,6089,3,3,141,91-180,4,20,0-30,3500+,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9,1,9.440023,2.484907,2.484907,0.0,1,0,1
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,0.0,0.0,unknown,unknown,11751,58,87,78,75,1,0,3,88,51,3626,22,35,4206,17,26,463,365+,6,22,0-30,unknown,unknown,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8,0,9.371779,4.077537,4.369448,0.0,1,0,1
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,unknown,gemini-3-flash-preview,19140,24,177,145,144,0,0,43,88,33,4211,10,14,6452,2,9,263,181-365,5,14,0-30,2000-3500,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7,1,9.859588,3.218876,4.983607,0.0,1,0,1


In [3]:
features = [
    "avg_position",
    "content_age_days",
    "engagement_rate",
    "ctr",
    "trend_pct"
]

target = "is_declining_label"

X = df[features]

y = df[target].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [4]:
df["prediction_probability"] = model.predict_proba(X)[:,1]

In [5]:
def assign_action(score):

    if score >= 0.80:
        return "Refresh"

    elif score >= 0.50:
        return "Review"

    else:
        return "Leave"

In [6]:
df["action"] = df["prediction_probability"].apply(assign_action)

In [7]:
def reason(row):

    if row["content_age_days"] > 365:
        return "Old content"

    elif row["avg_position"] > 20:
        return "Low ranking"

    elif row["trend_pct"] < -10:
        return "Traffic decline"

    else:
        return "General review"

In [8]:
df["reason_code"] = df.apply(reason, axis=1)

In [9]:
queue = df[
    [
        "content_id",
        "prediction_probability",
        "action",
        "reason_code",
        "avg_position",
        "content_age_days",
        "ctr",
        "trend_pct"
    ]
].sort_values(
    "prediction_probability",
    ascending=False
)

queue.head(20)

,content_id,prediction_probability,action,reason_code,avg_position,content_age_days,ctr,trend_pct
14,content_91067a14431a,1.0,Refresh,Low ranking,27.1,124,0.00,-64.9
9,content_c27558df2b0c,1.0,Refresh,Traffic decline,4.9,257,0.16,-29.2
8,content_5e6c160719bc,1.0,Refresh,Low ranking,46.0,90,0.09,-58.8
6,content_9a34b442b552,1.0,Refresh,Traffic decline,7.0,90,0.00,-92.3
5,content_d4084a4bc775,1.0,Refresh,Traffic decline,8.5,147,0.03,-38.9
4,content_d99b7a2d90ca,1.0,Refresh,Low ranking,44.0,263,0.13,-34.7
2,content_9aa793d4d895,1.0,Refresh,Low ranking,36.5,141,0.09,-60.9
1,content_a1fb4e703a9e,1.0,Refresh,Old content,20.3,445,0.05,-57.7
0,content_304f48230142,1.0,Refresh,Traffic decline,10.6,187,0.76,-41.4
28,content_19ad8f9bac29,1.0,Refresh,Low ranking,59.3,96,0.00,-60.0


In [10]:
queue.head(20).to_csv(

    "C:/Users/ok/Documents/FlyRank Internship/Week 1/flyrank-ml-internship-starter-main/work/outputs/action_queue.csv",

    index=False

)

print("Action queue exported.")

Action queue exported.


In [12]:
df["prediction_probability"].describe()

count    30000.000000
mean         0.542072
std          0.497920
min          0.000000
25%          0.000000
50%          1.000000
75%          1.000000
max          1.000000
Name: prediction_probability, dtype: float64

# Intended Use

This playbook is designed to help SEO and content teams prioritize pages for review and refresh.

The model ranks pages according to their likelihood of declining performance.

Editors should use these recommendations to identify high-priority pages while making the final decision through human review.

# Ranked Actions

The machine learning model generates three action categories.

| Action | Meaning |
|---------|----------|
| Refresh | Update the content immediately |
| Review | Inspect before making changes |
| Leave | No immediate action required |

The pages with the highest prediction probabilities appear at the top of the exported action queue.

# Human Review Checklist

Before applying any recommendation, a human reviewer should verify:

- Content accuracy
- Outdated information
- Search intent
- Keyword relevance
- Broken links
- Competitor changes
- Grammar and readability

The model supports decision-making but does not replace human judgment.

# Intended Use and Limitations

The playbook should be used as a prioritization tool rather than an automatic publishing system.

Limitations include:

- Seasonal content may be incorrectly flagged.
- External events can affect traffic unexpectedly.
- The model is trained using historical data and may require periodic retraining.
- Human review remains essential before implementing recommendations.

# Monitoring and Retraining

The model should be monitored continuously.

Retraining is recommended when:

- Search behavior changes significantly.
- Google releases major algorithm updates.
- New content types become available.
- Prediction accuracy decreases.
- Feature distributions shift over time.

In [11]:
importance = pd.DataFrame({

    "Feature":features,

    "Importance":model.feature_importances_

})

importance.sort_values(

    "Importance",

    ascending=False

)

,Feature,Importance
4,trend_pct,0.962559
0,avg_position,0.021757
1,content_age_days,0.012785
3,ctr,0.002440
2,engagement_rate,0.000459


# Feature Importance Interpretation

The model identified **trend_pct** as the strongest predictor of declining content.

Other important features include:

- Average Position
- Content Age
- CTR
- Engagement Rate

These features help the model prioritize pages that are most likely to benefit from content updates.

# Top 10 Action Review

The highest-ranked pages should be manually inspected.

Typical reasons include:

- Older content requiring updates.
- Declining search trends.
- Poor average rankings.
- Low click-through rates.

These recommendations help content teams focus their effort on pages with the greatest opportunity for improvement.

# Conclusion

This notebook transformed the validated Random Forest model into a practical content action playbook.

The model predictions were converted into ranked actions, reason codes, and an exportable action queue for content teams.

Human review remains an essential part of the workflow to ensure recommendations are appropriate before implementation.

This playbook demonstrates how machine learning can support SEO decision-making while maintaining responsible human oversight.

## Observation: 
The prediction probabilities are concentrated at 0 and 1. Combined with the very high accuracy (99.98%), this suggests the model may be overfitting or that the target variable is highly predictable from the selected features. In a production environment, additional validation, cross-validation, and leakage checks should be performed before deployment.